In [3]:
from tradepy.data.loader import load_future
from tradepy.config.config import load_config, load_symbols, load_feature_config, load_exclude_config
from tradepy.data.cleaner import add_trading_date_by_gap
from tradepy.data.resampler import resample_ohlcv, daily_ohlcv_cummulative
from tradepy.features.generate import generate_features
from tradepy.features.quality_check import data_quality_report
from tradepy.supervised.load import prepare_all
from tradepy.paths import EXCLUDE_CONFIG, SYSTEM_CONFIG
from tradepy.supervised.target import target_triple_barrier_interday
from tradepy.supervised.pretrain import filter_features
from tradepy.supervised.models import GoldLSTM_L1_Move, GoldLSTM_L2_Dir, GoldGRU_L1_Move, GoldGRU_L2_Dir, BasicLSTM_L1_Move, BasicLSTM_L2_Dir, BasicGRU_L1_Move, BasicGRU_L2_Dir, BasicLSTM_L3_Regression, BasicGRU_L3_Regression
from tradepy.supervised.train import train_walk_forward_2level, grid_search_thresholds, create_lstm_dataset
from tradepy.supervised.posttrain import evaluate_model_classification, tune_threshold_wf, save_trading_model_2level
from tradepy.paths import MODELS_DIR

In [4]:
symbols = load_symbols()
cfg = load_config()

In [5]:
ASSET = symbols[5]
MINUTES             = cfg['sampling_minutes']        # 240
RETURN_HORIZON_MIN  = cfg['return_horizon_min']      # 2880

In [8]:
import os
import torch
import joblib
import pandas as pd

def load_trading_model_3level(
    model_class_l1,
    model_class_l2,
    model_class_l3,
    path="trading_model_3level",
    device="cpu"
):
    """
    Carga un pipeline completo de 3 niveles:
        - L1: Movimiento
        - L2: Dirección
        - L3: Regresión
    Devuelve:
        model_l1, model_l2, model_l3,
        scaler_features, scaler_l3,
        features_list, model_params,
        best_thresholds, full_results
    """

    # -----------------------------
    # 1. Cargar metadatos
    # -----------------------------
    model_params     = joblib.load(os.path.join(path, "model_params.pkl"))
    features_list    = joblib.load(os.path.join(path, "features.pkl"))
    scaler_features  = joblib.load(os.path.join(path, "scaler.pkl"))
    scaler_l3        = joblib.load(os.path.join(path, "scaler_l3.pkl"))

    thresholds_path = os.path.join(path, "best_thresholds.pkl")
    best_thresholds = joblib.load(thresholds_path) if os.path.exists(thresholds_path) else None

    # -----------------------------
    # 2. Cargar modelos
    # -----------------------------
    model_l1 = model_class_l1(**model_params)
    model_l1.load_state_dict(torch.load(os.path.join(path, "model_l1_weights.pth"), map_location=device))
    model_l1.to(device)
    model_l1.eval()

    model_l2 = model_class_l2(**model_params)
    model_l2.load_state_dict(torch.load(os.path.join(path, "model_l2_weights.pth"), map_location=device))
    model_l2.to(device)
    model_l2.eval()

    model_l3 = model_class_l3(input_dim=model_params["input_dim"], output_dim=1)
    model_l3.load_state_dict(torch.load(os.path.join(path, "model_l3_weights.pth"), map_location=device))
    model_l3.to(device)
    model_l3.eval()

    # -----------------------------
    # 3. Cargar resultados parquet
    # -----------------------------
    parquet_path = os.path.join(path, "res.parquet")
    full_results = pd.read_parquet(parquet_path) if os.path.exists(parquet_path) else None

    print(f"🚀 Pipeline 3-Level cargado correctamente desde: {path}")
    print(f"📌 Features: {len(features_list)}")
    print(f"📌 Thresholds cargados: {best_thresholds is not None}")
    print(f"📁 Resultados cargados: {full_results is not None}")

    return (
        model_l1,
        model_l2,
        model_l3,
        scaler_features,
        scaler_l3,
        features_list,
        model_params,
        best_thresholds,
        full_results
    )


In [9]:
(
    model_l1,
    model_l2,
    model_l3,
    sc_features,
    sc_l3,
    features,
    params,
    thresholds,
    df_results
) = load_trading_model_3level(
        BasicLSTM_L1_Move,
        BasicLSTM_L2_Dir,
        BasicLSTM_L3_Regression,
        path=f"{MODELS_DIR}/{ASSET.lower()}/trading_model_lstm_3level",
        device="cuda"
)


🚀 Pipeline 3-Level cargado correctamente desde: E:\Futuro\MLAlgoTrading\TradingBots\Models/gc/trading_model_lstm_3level
📌 Features: 71
📌 Thresholds cargados: True
📁 Resultados cargados: True
